# Pregunta 9 — Evolución mensual de la producción

**Pregunta de investigación:** ¿Cómo evolucionaron los litros procesados y kilos obtenidos por mes y masa?

**Fuente:** `registro_produccion_LIMPIO (4).xlsx`, hoja `Registro Produccion`. Se utiliza el archivo original **solo en lectura**. Los precios ya fueron divididos por 5 en origen y **no se vuelven a dividir**.

**Objetivo del análisis:** Identificar meses máximos y mínimos de producción y participación de cada masa.

**Indicador:** Indicadores calculados en las celdas siguientes, con unidad y período declarados.


## 1. Contexto
Los totales anuales pueden ocultar meses con picos o caídas de producción.

## 2. Gancho
**¿Qué puede sorprender?** La diferencia entre elaboraciones, grupos o períodos podría ser mayor de lo que sugiere mirar únicamente un promedio general. La magnitud exacta se establece con los datos, no se supone.

## 3. Preparación reproducible
Las transformaciones se calculan en memoria. No se modifica ninguna celda del Excel original.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict, Counter
from artifact_tool import Blob, SpreadsheetFile
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, mean_absolute_error, r2_score, accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
SEMILLA=42
ruta=Path('registro_produccion_LIMPIO (4).xlsx')
if not ruta.exists(): ruta=Path('/mnt/data/registro_produccion_LIMPIO (4).xlsx')
if not ruta.exists(): raise FileNotFoundError('Colocá el Excel junto al notebook')
w=SpreadsheetFile.import_xlsx(Blob.load(str(ruta)))
mat=w.worksheets.get_item_at(0).get_range('A1:CA1112').values
headers=mat[0]; registros=mat[1:]; ix={h:i for i,h in enumerate(headers)}
def valor(r,col):
    x=r[ix[col]]
    try: return float(x) if x is not None and x!='' else np.nan
    except (ValueError,TypeError): return np.nan
def matriz(cols,filas=registros): return np.array([[valor(r,c) for c in cols] for r in filas],float)
def tasas(r):
    l=valor(r,'LITROS DE LECHE')
    return [valor(r,c)*1000/l if np.isfinite(l) and l>0 else np.nan for c in ['AZUCAR','SUERO','BICARBONATO','L.P.D']]
def features():return np.array([[valor(r,'LITROS DE LECHE'),*tasas(r),valor(r,'MATERIA GRASA'),valor(r,'PROTEINAS')] for r in registros],float)
X=features();masas=np.array([str(r[ix['MASA']]) for r in registros]);rend=matriz(['RENDIMIENTO %']).ravel();kg=matriz(['TOTAL KG OBTENIDOS']).ravel()
def normalizar(X):return Pipeline([('imputar',SimpleImputer(strategy='median')),('escalar',StandardScaler())]).fit_transform(X)
def grafico(titulo,unidad,transformacion):
    plt.title(titulo,loc='left',fontweight='bold',fontsize=13)
    plt.figtext(.01,.005,f'Fuente: Registro Produccion | {unidad} | {transformacion} | N={len(registros)}',fontsize=8,color='dimgray')
    plt.tight_layout(rect=(0,.055,1,1));plt.show()
print('Archivo leído en modo solo lectura:',len(registros),'elaboraciones; ',len(headers),'columnas')


## 4. Hallazgos y gráficos
Los gráficos se eligen por la relación que se quiere mostrar: ranking, distribución, comparación, correlación o evolución temporal. Las figuras llevan fuente, unidad y transformación. Los resultados que aparecen a continuación se calculan al ejecutar las celdas.

In [ ]:
mensual=defaultdict(lambda:{'litros':0.,'kg':0.,'masas':defaultdict(float),'n':0})
for r in registros:
    a=valor(r,'AÑO');m=valor(r,'MES');l=valor(r,'LITROS DE LECHE');k=valor(r,'TOTAL KG OBTENIDOS')
    if not all(np.isfinite(z) for z in [a,m,l,k]):continue
    key=f'{int(a)}-{int(m):02d}';mensual[key]['litros']+=l;mensual[key]['kg']+=k;mensual[key]['masas'][str(r[ix['MASA']])]+=k;mensual[key]['n']+=1
meses=sorted(mensual);lit=[mensual[z]['litros'] for z in meses];kgs=[mensual[z]['kg'] for z in meses]
fig,ax=plt.subplots(figsize=(12,5));ax.plot(meses,kgs,marker='o');ax.set_ylabel('Kilogramos obtenidos');ax.tick_params(axis='x',rotation=60);grafico('Evolución mensual de kilos producidos','kg','suma de elaboraciones por mes')
fig,ax=plt.subplots(figsize=(12,5));ax.plot(meses,lit,marker='o');ax.set_ylabel('Litros procesados');ax.tick_params(axis='x',rotation=60);grafico('Evolución mensual de leche procesada','litros','suma por mes; verificar meses incompletos')
orden=sorted(set(masas));bottom=np.zeros(len(meses));fig,ax=plt.subplots(figsize=(12,5))
for z in orden:
    y=np.array([mensual[mes]['masas'][z] for mes in meses]);ax.bar(meses,y,bottom=bottom,label=z);bottom+=y
ax.legend(ncol=4);ax.set_ylabel('Kilogramos obtenidos');ax.tick_params(axis='x',rotation=60);grafico('Aporte de cada masa a la producción mensual','kg','sumas mensuales por tipo de masa')
print('COMPRENSIÓN: mes máximo',meses[int(np.argmax(kgs))],round(max(kgs),1),'kg; mínimo',meses[int(np.argmin(kgs))],round(min(kgs),1),'kg. Verificar cobertura de meses extremos.')


## 5. Comprensión: insight central
Tomá la frase que imprime la celda anterior como base de la conclusión. **No afirmes causas** que los datos no permitan demostrar. Explicá qué se observa, cuánto representa y qué limitaciones tiene.

## 6. Acción propuesta
Analizar los meses extremos junto con días operativos, turnos y demanda, que este dataset no permite comprobar.

## 7. Control ético y límites
- Fuente: registro de producción suministrado por la empresa. Período: el cubierto por el Excel; sin recortar meses intencionalmente.
- Indicá unidades, cantidad de registros válidos y transformaciones aplicadas.
- Los gráficos de barras parten de cero; los gráficos de distribución y comparación no ocultan categorías.
- No se cambian ni completan los datos originales; cualquier imputación ocurre únicamente en el modelo.
- La interpretación se limita a asociaciones y predicciones, no a causalidad comprobada.
